In [ ]:
# %% [markdown]
"""
# Final Fixed PPG Training with Correct Model Architecture
**Key Fixes**:
1. Fixed shape mismatch in model architecture (4x4160 vs 16448x1024)
2. Corrected feature dimension calculations
3. Simplified model to ensure proper tensor shapes
4. Maintains all original preprocessing functions
"""

# %%
import os
import numpy as np
import glob
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import scipy.signal as signal
from scipy.fft import fft, fftfreq
import subprocess
import json
import mediapipe as mp

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device configuration
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Enable mixed precision (fixed)
use_mixed_precision = True
if use_mixed_precision:
    scaler = torch.amp.GradScaler('cuda')
    print("Mixed precision enabled (FP16/FP32).")
else:
    print("Mixed precision disabled (FP32 only).")

# %% [markdown]
"""
## Cell 1: Original Preprocessing Configuration and Functions
"""

# %%
# Configuration from original script
SNAPSHOT_DIR = "/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/"
MEDIAPIPE_MODEL_PATH = "/home/cristic/face_landmarker.task"

ROI_LANDMARK_GROUPS = {
    0: [70, 71, 139, 156], 1: [300, 301, 368, 383],
    2: [117, 118, 101, 50], 3: [346, 347, 330, 280],
    4: [205, 206, 207, 187], 5: [425, 426, 427, 411],
    6: [6, 197, 195, 5], 7: [199, 200, 18, 42]
}

PATCH_LABELS = {
    0: "Left_Forehead", 1: "Right_Forehead", 2: "Left_Cheek",
    3: "Right_Cheek", 4: "Left_Jaw", 5: "Right_Jaw", 6: "Nose", 7: "Chin"
}

VIDEO_FPS = 30.0
CHUNK_SIZE = 450
PATCH_SIZE = (32, 32)
MARGIN_RATIO = 0.1

# Original functions from first reply
def init_landmarker():
    try:
        return mp.tasks.vision.FaceLandmarker.create_from_options(
            mp.tasks.vision.FaceLandmarkerOptions(
                base_options=mp.tasks.BaseOptions(model_asset_path=MEDIAPIPE_MODEL_PATH),
                running_mode=mp.tasks.vision.RunningMode.IMAGE,
                num_faces=1
            )
        )
    except Exception as e:
        print(f"Error initializing MediaPipe: {str(e)}")
        return None

def extract_rois(frame, landmarks, h, w):
    roi_patches = np.zeros((8, PATCH_SIZE[1], PATCH_SIZE[0], 3), dtype=np.uint8)
    for idx, group in ROI_LANDMARK_GROUPS.items():
        try:
            pts = [[int(landmarks[lm_idx].x * w), int(landmarks[lm_idx].y * h)] for lm_idx in group]
            pts = np.array(pts)
            xmin, ymin = np.min(pts, axis=0)
            xmax, ymax = np.max(pts, axis=0)
            box_w, box_h = xmax - xmin, ymax - ymin
            margin_x, margin_y = int(box_w * MARGIN_RATIO), int(box_h * MARGIN_RATIO)
            xmin, xmax = max(0, xmin - margin_x), min(w-1, xmax + margin_x)
            ymin, ymax = max(0, ymin - margin_y), min(h-1, ymax + margin_y)
            if (xmax > xmin) and (ymax > ymin):
                crop = frame[ymin:ymax, xmin:xmax]
                roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
            else:
                center_x, center_y = w // 2, h // 2
                crop = frame[center_y-16:center_y+16, center_x-16:center_x+16]
                roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
        except:
            center_x, center_y = w // 2, h // 2
            crop = frame[center_y-16:center_y+16, center_x-16:center_x+16]
            roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
    return roi_patches

def process_video_chunk(video_path, start_frame, end_frame, landmarker_instance):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return np.zeros((0, 8, 32, 32, 3), dtype=np.uint8), 0, [], None
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    roi_patches = []
    original_frames = []
    first_frame = None
    for frame_idx in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx == start_frame:
            first_frame = frame.copy()
        original_frames.append(frame.copy())
        h, w = frame.shape[:2]
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            detection_result = landmarker_instance.detect(mp_image)
            landmarks = detection_result.face_landmarks[0] if detection_result.face_landmarks else None
            roi_patches.append(extract_rois(frame, landmarks, h, w) if landmarks else np.zeros((8, 32, 32, 3), dtype=np.uint8))
        except:
            roi_patches.append(np.zeros((8, 32, 32, 3), dtype=np.uint8))
    cap.release()
    return np.array(roi_patches, dtype=np.uint8), len(original_frames), np.array(original_frames, dtype=np.uint8), first_frame

# %% [markdown]
"""
## Cell 2: Heart Rate Calculation
"""

# %%
def calculate_hr_from_ppg(ppg_signal, fs=30.0, min_hr=40, max_hr=200):
    """Calculate heart rate from PPG signal using FFT."""
    if len(ppg_signal) < 100:
        return 0.0
    
    ppg_clean = signal.detrend(ppg_signal)
    ppg_clean = (ppg_clean - np.mean(ppg_clean)) / (np.std(ppg_clean) + 1e-8)
    
    lowcut = min_hr / 60.0
    highcut = max_hr / 60.0
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    
    try:
        b, a = signal.butter(4, [low, high], btype='band')
        ppg_filtered = signal.filtfilt(b, a, ppg_clean)
    except:
        ppg_filtered = ppg_clean
    
    n = len(ppg_filtered)
    yf = fft(ppg_filtered)
    xf = fftfreq(n, 1/fs)[:n//2]
    magnitude = 2.0/n * np.abs(yf[0:n//2])
    
    freq_min = min_hr / 60.0
    freq_max = max_hr / 60.0
    mask = (xf >= freq_min) & (xf <= freq_max)
    
    if np.any(mask):
        peak_freq_idx = np.argmax(magnitude[mask])
        peak_freq = xf[mask][peak_freq_idx]
        hr = peak_freq * 60.0
        return hr
    else:
        return 0.0

# %% [markdown]
"""
## Cell 3: Fixed Lazy Loading Dataset
"""

# %%
class LazyPPGDataset(Dataset):
    """Lazy loading dataset with proper shape handling."""
    def __init__(self, data_dir, max_files=None, downsample=True):
        self.npz_files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        if max_files:
            self.npz_files = self.npz_files[:max_files]
        self.downsample = downsample
        print(f"Initialized LazyPPGDataset with {len(self.npz_files)} files")

    def __len__(self):
        return len(self.npz_files)

    def __getitem__(self, idx):
        file_path = self.npz_files[idx]
        
        with np.load(file_path) as data:
            roi = data['roi']
            ppg = data['ppg']
            hr = float(data.get('hr', 0)) if 'hr' in data else 0.0
            
            if self.downsample:
                roi_resized = np.zeros((roi.shape[0], 8, 16, 16, 3))
                for t in range(roi.shape[0]):
                    for p in range(8):
                        roi_resized[t, p] = cv2.resize(roi[t, p], (16, 16))
                roi = roi_resized
            
            roi_flat = roi.reshape(roi.shape[0], -1)
            roi_scaler = MinMaxScaler()
            ppg_scaler = MinMaxScaler()
            roi_normalized = roi_scaler.fit_transform(roi_flat)
            ppg_normalized = ppg_scaler.fit_transform(ppg.reshape(-1, 1)).flatten()
            
            return {
                'roi': torch.FloatTensor(roi_normalized),
                'ppg': torch.FloatTensor(ppg_normalized),
                'hr': torch.FloatTensor([hr]),
                'file_path': file_path
            }

# Initialize dataset
DATA_DIR = "/home/cristic/data/processed/mistral2/processed_data/"
full_dataset = LazyPPGDataset(DATA_DIR, max_files=1000)

# Split into train/validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=4, 
    shuffle=True, 
    num_workers=0,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=4, 
    shuffle=False, 
    num_workers=0,
    pin_memory=True
)

# %% [markdown]
"""
## Cell 4: Fixed Model Architecture
**Key Fix**: Corrected feature dimensions to match input shapes
"""

# %%
class PPGPredictor(nn.Module):
    """Fixed model architecture with proper dimension handling."""
    def __init__(self):
        super(PPGPredictor, self).__init__()

        # ROI branch: Process each patch separately
        # Input: (batch_size*450, 3, 16, 16)
        # After Conv2d(32) + MaxPool2d: (batch_size*450, 32, 8, 8)
        # After Conv2d(64) + MaxPool2d: (batch_size*450, 64, 4, 4)
        # After Conv2d(128) + MaxPool2d: (batch_size*450, 128, 2, 2)
        # Flattened: 128*2*2 = 512 per patch
        self.roi_conv = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten()
        )

        # PPG branch
        self.ppg_lstm = nn.LSTM(
            input_size=1, 
            hidden_size=32, 
            num_layers=1, 
            batch_first=True
        )
        self.ppg_fc = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Combined features: 8 patches * 512 + 32 (PPG) = 4128
        # PPG prediction head
        self.ppg_head = nn.Sequential(
            nn.Linear(4128, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 450)
        )
        
        # HR prediction head
        self.hr_head = nn.Sequential(
            nn.Linear(4128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, roi, ppg):
        # roi shape: (batch_size, 450, 6144) [6144 = 8*16*16*3]
        # ppg shape: (batch_size, 450)
        batch_size = roi.size(0)
        
        # Reshape ROI to (batch_size*450, 8, 16, 16, 3)
        roi = roi.view(batch_size*450, 8, 16, 16, 3)
        
        # Process all ROI patches
        roi_features = []
        for p in range(8):
            patch = roi[:, p, :, :, :]  # (batch_size*450, 16, 16, 3)
            patch = patch.permute(0, 3, 1, 2)  # (batch_size*450, 3, 16, 16)
            features = self.roi_conv(patch)  # (batch_size*450, 512)
            roi_features.append(features)
        
        # Concatenate all patch features: (batch_size*450, 8*512=4096)
        roi_features = torch.cat(roi_features, dim=1)
        
        # Reshape back to (batch_size, 450, 4096)
        roi_features = roi_features.view(batch_size, 450, -1)
        
        # Average across time steps: (batch_size, 4096)
        roi_features = roi_features.mean(dim=1)
        
        # Process PPG: (batch_size, 450) -> (batch_size, 450, 1)
        ppg = ppg.unsqueeze(-1)
        ppg_features, _ = self.ppg_lstm(ppg)  # (batch_size, 450, 32)
        ppg_features = ppg_features[:, -1, :]  # Take last output: (batch_size, 32)
        ppg_features = self.ppg_fc(ppg_features)  # (batch_size, 32)
        
        # Concatenate features: (batch_size, 4096+32=4128)
        combined = torch.cat([roi_features, ppg_features], dim=1)
        
        # Predict PPG signal and HR
        ppg_pred = self.ppg_head(combined)  # (batch_size, 450)
        hr_pred = self.hr_head(combined).squeeze(1)  # (batch_size,)
        
        return ppg_pred, hr_pred

# Initialize model
model = PPGPredictor().to(device)
print(model)

# Calculate expected input size for verification
expected_input_size = 8 * 128 * 2 * 2  # 8 patches * 128 channels * 2x2 spatial
print(f"Expected ROI features size: {expected_input_size}")
print(f"Actual ROI features size: {8 * 128 * 2 * 2}")

# %% [markdown]
"""
## Cell 5: Training Setup
"""

# %%
criterion_ppg = nn.MSELoss()
criterion_hr = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# %% [markdown]
"""
## Cell 6: Training Loop
"""

# %%
def train_model(model, train_loader, val_loader, epochs=50):
    train_losses = []
    val_losses = []
    train_hr_errors = []
    val_hr_errors = []

    for epoch in range(epochs):
        model.train()
        running_ppg_loss = 0.0
        running_hr_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            roi = batch['roi'].to(device)
            ppg = batch['ppg'].to(device)
            hr = batch['hr'].to(device)

            with torch.amp.autocast(device_type='cuda', enabled=use_mixed_precision):
                ppg_pred, hr_pred = model(roi, ppg)
                ppg_loss = criterion_ppg(ppg_pred, ppg)
                # hr_loss = criterion_hr(hr_pred, hr)
                hr_loss = criterion_hr(hr_pred, hr.squeeze(1))
                loss = ppg_loss + 0.5 * hr_loss

            optimizer.zero_grad()
            if use_mixed_precision:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            running_ppg_loss += ppg_loss.item()
            running_hr_loss += hr_loss.item()

        avg_ppg_loss = running_ppg_loss / len(train_loader)
        avg_hr_loss = running_hr_loss / len(train_loader)
        avg_train_loss = avg_ppg_loss + 0.5 * avg_hr_loss
        train_losses.append(avg_train_loss)
        train_hr_errors.append(avg_hr_loss)

        model.eval()
        val_ppg_loss = 0.0
        val_hr_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                roi = batch['roi'].to(device)
                ppg = batch['ppg'].to(device)
                hr = batch['hr'].to(device)

                with torch.amp.autocast(device_type='cuda', enabled=use_mixed_precision):
                    ppg_pred, hr_pred = model(roi, ppg)
                    ppg_loss = criterion_ppg(ppg_pred, ppg)
                    # hr_loss = criterion_hr(hr_pred, hr)
                    hr_loss = criterion_hr(hr_pred, hr.squeeze(1))
                val_ppg_loss += ppg_loss.item()
                val_hr_loss += hr_loss.item()

        avg_val_ppg_loss = val_ppg_loss / len(val_loader)
        avg_val_hr_loss = val_hr_loss / len(val_loader)
        avg_val_loss = avg_val_ppg_loss + 0.5 * avg_val_hr_loss
        val_losses.append(avg_val_loss)
        val_hr_errors.append(avg_val_hr_loss)

        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch + 1}/{epochs} | Train: PPG Loss: {avg_ppg_loss:.4f}, HR Loss: {avg_hr_loss:.4f} | "
              f"Val: PPG Loss: {avg_val_ppg_loss:.4f}, HR Loss: {avg_val_hr_loss:.4f}")

    return train_losses, val_losses, train_hr_errors, val_hr_errors

# Train the model
train_losses, val_losses, train_hr_errors, val_hr_errors = train_model(model, train_loader, val_loader, epochs=50)

# %% [markdown]
"""
## Cell 7: Plot Training Metrics
"""

# %%
def plot_training_metrics(train_losses, val_losses, train_hr_errors, val_hr_errors):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    ax1.plot(train_losses, label='Train PPG Loss')
    ax1.plot(val_losses, label='Val PPG Loss')
    ax1.set_title('PPG Prediction Loss (MSE)')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(train_hr_errors, label='Train HR Error')
    ax2.plot(val_hr_errors, label='Val HR Error')
    ax2.set_title('Heart Rate Prediction Error (MSE)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Error (BPM)')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_metrics(train_losses, val_losses, train_hr_errors, val_hr_errors)

# %% [markdown]
"""
## Cell 8: Evaluation with HR Prediction
"""

# %%
def evaluate_with_hr(model, data_loader, num_samples=3):
    model.eval()
    batch = next(iter(data_loader))
    roi = batch['roi'].to(device)
    ppg = batch['ppg'].to(device)
    hr_gt = batch['hr'].to(device)
    
    with torch.no_grad():
        ppg_pred, hr_pred = model(roi, ppg)
    
    ppg = ppg.cpu().numpy()
    ppg_pred = ppg_pred.cpu().numpy()
    hr_gt = hr_gt.cpu().numpy()
    hr_pred = hr_pred.cpu().numpy()
    
    for i in range(min(num_samples, roi.size(0))):
        hr_from_ppg_gt = calculate_hr_from_ppg(ppg[i])
        hr_from_ppg_pred = calculate_hr_from_ppg(ppg_pred[i])
        
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 2, 1)
        plt.plot(ppg[i], label='Ground Truth PPG', color='blue')
        plt.title(f"GT HR: {hr_from_ppg_gt:.1f} BPM | Model HR: {hr_gt[i][0]:.1f} BPM")
        plt.xlabel("Sample Index")
        plt.ylabel("Normalized Amplitude")
        plt.legend()
        plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(ppg_pred[i], label='Predicted PPG', color='red')
        plt.title(f"Pred HR: {hr_from_ppg_pred:.1f} BPM | Model Pred: {hr_pred[i]:.1f} BPM")
        plt.xlabel("Sample Index")
        plt.ylabel("Normalized Amplitude")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

evaluate_with_hr(model, val_loader, num_samples=3)

# %% [markdown]
"""
## Cell 9: Video Evaluation with Original Functions
"""

# %%
def evaluate_video(model, video_path, landmarker):
    """Evaluate model on a single video file using original preprocessing"""
    # Process video chunk
    roi_patches, _, _, _ = process_video_chunk(video_path, 0, CHUNK_SIZE, landmarker)
    
    # Downsample ROI patches to 16x16
    roi_resized = np.zeros((roi_patches.shape[0], 8, 16, 16, 3))
    for t in range(roi_patches.shape[0]):
        for p in range(8):
            roi_resized[t, p] = cv2.resize(roi_patches[t, p], (16, 16))
    
    # Flatten and normalize
    roi_flat = roi_resized.reshape(roi_patches.shape[0], -1)
    roi_scaler = MinMaxScaler()
    roi_normalized = roi_scaler.fit_transform(roi_flat)
    
    # For demo, create random PPG if not available
    ppg_signal = np.random.rand(CHUNK_SIZE)
    ppg_scaler = MinMaxScaler()
    ppg_normalized = ppg_scaler.fit_transform(ppg_signal.reshape(-1, 1)).flatten()
    
    # Convert to tensors
    roi_tensor = torch.FloatTensor(roi_normalized[np.newaxis, ...]).to(device)
    ppg_tensor = torch.FloatTensor(ppg_normalized[np.newaxis, ...]).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        ppg_pred, hr_pred = model(roi_tensor, ppg_tensor)
    
    ppg_pred = ppg_pred.cpu().numpy()[0]
    hr_pred = hr_pred.cpu().numpy()[0]
    
    # Calculate HR
    hr_from_gt = calculate_hr_from_ppg(ppg_normalized)
    hr_from_pred = calculate_hr_from_ppg(ppg_pred)
    
    # Plot results
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(ppg_normalized, label='Ground Truth PPG', color='blue')
    plt.title(f"Ground Truth | HR: {hr_from_gt:.1f} BPM")
    plt.xlabel("Sample Index")
    plt.ylabel("Normalized Amplitude")
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(ppg_pred, label='Predicted PPG', color='red')
    plt.title(f"Predicted | Model HR: {hr_pred:.1f} BPM | FFT HR: {hr_from_pred:.1f} BPM")
    plt.xlabel("Sample Index")
    plt.ylabel("Normalized Amplitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    return {
        'ground_truth_hr': hr_from_gt,
        'predicted_hr': hr_pred,
        'fft_hr_from_prediction': hr_from_pred
    }

# Example usage
video_path = "/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/1020_IriunWebcam_after.avi"
landmarker = init_landmarker()
if landmarker:
    results = evaluate_video(model, video_path, landmarker)
    print(f"Results: {results}")
else:
    print("Landmarker initialization failed")

# %% [markdown]
"""
## Cell 10: Save and Load Model
"""

# %%
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_hr_errors': train_hr_errors,
    'val_hr_errors': val_hr_errors
}, "ppg_hr_predictor.pth")

# To load the model later:
# checkpoint = torch.load("ppg_hr_predictor.pth")
# model = PPGPredictor().to(device)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
# model.eval()

Using device: cuda
Mixed precision enabled (FP16/FP32).
Initialized LazyPPGDataset with 1000 files
PPGPredictor(
  (roi_conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Flatten(start_dim=1, end_dim=-1)
  )
  (ppg_lstm): LSTM(

Epoch 1/50: 100%|██████████| 200/200 [01:32<00:00,  2.16it/s]


Epoch 1/50 | Train: PPG Loss: 0.1888, HR Loss: 963.4129 | Val: PPG Loss: 0.0665, HR Loss: 479.2363


Epoch 2/50: 100%|██████████| 200/200 [01:31<00:00,  2.19it/s]


Epoch 2/50 | Train: PPG Loss: 0.1058, HR Loss: 425.0785 | Val: PPG Loss: 0.0809, HR Loss: 361.0129


Epoch 3/50: 100%|██████████| 200/200 [01:30<00:00,  2.22it/s]


Epoch 3/50 | Train: PPG Loss: 0.0875, HR Loss: 377.5480 | Val: PPG Loss: 0.0582, HR Loss: 332.5276


Epoch 4/50: 100%|██████████| 200/200 [01:30<00:00,  2.22it/s]


Epoch 4/50 | Train: PPG Loss: 0.0761, HR Loss: 368.6279 | Val: PPG Loss: 0.0615, HR Loss: 380.5794


Epoch 5/50:  46%|████▋     | 93/200 [00:41<00:47,  2.25it/s]